In [ ]:
# neue pdfs in pdf_info.json aufnehmen
import json
import os

# Paths
json_path = r"C:\Workspace\BCE\up19040\gist-lmu-bbk\data\docs\pdf_info.json"
json_path_new = r"C:\Workspace\BCE\up19040\gist-lmu-bbk\data\docs\pdf_info_new.json"
pdf_dir = r"R:\Zentrale\Projekte\Tresor\Tresor80\IFRS"
json_prefix = "./data/pdfs/"
sample = "sample_M13_IFRS_20250909"

# Load existing JSON
with open(json_path, encoding="utf-8") as f:
    data = json.load(f)

# Get all PDF files in the directory
pdf_files = [f for f in os.listdir(pdf_dir) if f.lower().endswith(".pdf")]
print(len(pdf_files))

# Add new entries if not already present, else update in_sample if needed
for pdf_file in pdf_files:
    key = f"{json_prefix}{pdf_file}"
    if key not in data:
        data[key] = {"in_sample": [sample], "in_gold_standard": []}
    else:
        # Prüfen, ob sample_M13_IFRS_20250909 schon in in_sample ist
        if "in_sample" not in data[key]:
            data[key]["in_sample"] = []
        if sample not in data[key]["in_sample"]:
            data[key]["in_sample"].append(sample)
            print(f"{key}: {sample} ergänzt")
        # else:
        # print(f"{key}: sample_M13_IFRS_20250909 exists already")

# Save the updated JSON
with open(json_path_new, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4, ensure_ascii=False)

print("JSON updated successfully.")


In [ ]:
# divide large sample in pdf_info.json in n smaller samples
import json
import math

# Parameter
json_path = r"C:\Workspace\BCE\up19040\gist-lmu-bbk\data\docs\pdf_info.json"
target_sample = "sample_M13_IFRS_20250909"
n_parts = 3

# Lade die JSON-Datei
with open(json_path, encoding="utf-8") as f:
    data = json.load(f)

# Finde alle Keys, die das target_sample enthalten
matching_keys = [k for k, v in data.items() if target_sample in v.get("in_sample", [])]

# Gruppengröße berechnen
group_size = math.ceil(len(matching_keys) / n_parts)

# Verteile die Keys auf die Gruppen
for idx, key in enumerate(matching_keys):
    part_num = idx // group_size + 1
    part_name = f"{target_sample}-part{part_num}"
    if part_name not in data[key]["in_sample"]:
        data[key]["in_sample"].append(part_name)

# Speichere die Datei
output_path = json_path.replace(".json", "_grouped.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"Neue Datei gespeichert unter: {output_path}")


In [ ]:
# Anzahl der Einträge zu einem Sample zählen
import json

# Pfad zur JSON-Datei
json_path = r"C:\Workspace\BCE\up19040\gist-lmu-bbk\data\docs\pdf_info_grouped.json"

# Gesuchte Sample-ID
target_sample = "sample_M13_IFRS_20250909-part3"

# Datei laden
with open(json_path, encoding="utf-8") as f:
    data = json.load(f)

# Zählen
count = 0
for entry in data.values():
    if target_sample in entry.get("in_sample", []):
        count += 1

print(f"Anzahl der Einträge mit '{target_sample}' in 'in_sample': {count}")


In [5]:
# rename pdfs in an existing sample

import pandas as pd
import json 
import os

path_gs = r"C:\Users\up19040\Projekte\LMU\Goldstandards\gold_standard_with_ISINs_edited.csv"
df_gs = pd.read_csv(path_gs)

# Load the JSON file
json_path = r'C:\Users\up19040\Projekte\LMU\Programmieren mit LMU\ClimXtract\data\docs\pdf_info.json'
with open(json_path, 'r', encoding='utf-8') as f:
    pdf_info = json.load(f)

print(f"Original JSON has {len(pdf_info)} entries")
print("Sample JSON keys:", list(pdf_info.keys())[:3])

# DEBUG: Check df_gs content
print("\ndf_gs sample:")
print(df_gs[['report_name_old', 'report_name']].head())
print("\ndf_gs unique pairs:")
unique_pairs = df_gs[['report_name_old', 'report_name']].dropna().drop_duplicates()
print(unique_pairs.head())
print(f"Total unique pairs: {len(unique_pairs)}")

# Create mapping - FIX: Extract filename WITHOUT .pdf from JSON keys
json_filenames = {os.path.basename(k) for k in pdf_info.keys()}  # nur Dateinamen
print(f"JSON filenames sample: {list(json_filenames)[:3]}")

mapping = {}
for _, row in unique_pairs.iterrows():
    old_name = str(row['report_name_old']).strip()
    new_name = str(row['report_name']).strip()
    
    # Versuche beide Varianten: mit und ohne .pdf
    old_pdf = old_name if old_name.endswith('.pdf') else old_name + '.pdf'
    
    if (old_name != new_name and 
        old_pdf in json_filenames):
        mapping[old_pdf] = new_name + '.pdf' if not new_name.endswith('.pdf') else new_name
        print(f"MATCH FOUND: {old_pdf} -> {mapping[old_pdf]}")

print(f"\nMapping created: {len(mapping)} entries")
print("Sample mappings:", dict(list(mapping.items())[:5]))

# Update JSON keys
updated_pdf_info = {}
updated_count = 0

# Mapping für volle Pfade erstellen
full_path_mapping = {}
for old_pdf, new_pdf in mapping.items():
    for full_path in pdf_info.keys():
        if os.path.basename(full_path) == old_pdf:
            full_path_mapping[full_path] = f"./data/pdfs/{new_pdf}"
            break

print(f"Full path mappings: {len(full_path_mapping)}")

for old_key, value in pdf_info.items():
    if old_key in full_path_mapping:
        new_key = full_path_mapping[old_key]
        updated_pdf_info[new_key] = value
        updated_count += 1
        print(f"UPDATED: {old_key} -> {new_key}")
    else:
        updated_pdf_info[old_key] = value

# Save new JSON
new_json_path = json_path.replace('pdf_info.json', 'pdf_info_updated.json')
with open(new_json_path, 'w', encoding='utf-8') as f:
    json.dump(updated_pdf_info, f, indent=4)

print(f"\n✓ Updated {updated_count} entries")
print(f"✓ New JSON: {new_json_path}")
print(f"✓ Total entries: {len(updated_pdf_info)}")


Original JSON has 1046 entries
Sample JSON keys: ['./data/pdfs/aareal bank ag_2018_report.pdf', './data/pdfs/Abbvie_2019_en.pdf', './data/pdfs/acuity brands inc_2022_report.pdf']

df_gs sample:
           report_name_old              report_name
0  Allianz_2022_report.pdf  Allianz_2022_report.pdf
1  Allianz_2022_report.pdf  Allianz_2022_report.pdf
2  Allianz_2022_report.pdf  Allianz_2022_report.pdf
3  Allianz_2022_report.pdf  Allianz_2022_report.pdf
4  Allianz_2022_report.pdf  Allianz_2022_report.pdf

df_gs unique pairs:
                       report_name_old                        report_name
0              Allianz_2022_report.pdf            Allianz_2022_report.pdf
49             Daimler_2020_report.pdf            Daimler_2020_report.pdf
89        Fresenius SE_2019_report.pdf       Fresenius SE_2019_report.pdf
129  acuity brands inc_2022_report.pdf  acuity brands inc_2022_report.pdf
171            addtech_2022_report.pdf            addtech_2022_report.pdf
Total unique pairs: 139
JSON 

In [ ]:
# alt

import os
import json

# Paths
json_path = r'C:\Users\up19040\Projekte\LMU\Programmieren mit LMU\information-extraction-pilot\data\docs\pdf_info.json'
json_path_new = r'C:\Users\up19040\Projekte\LMU\Programmieren mit LMU\information-extraction-pilot\data\docs\pdf_info_new.json'

pdf_dir = r'C:\Users\up19040\Projekte\LMU\Programmieren mit LMU\information-extraction-pilot\data\pdfs\Anpassungspläne N1'
json_prefix = './data/pdfs/'

# Load existing JSON
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Get all PDF files in the directory
pdf_files = [f for f in os.listdir(pdf_dir) if f.lower().endswith('.pdf')]

# Add new entries if not already present
for pdf_file in pdf_files:
    key = f"{json_prefix}{pdf_file}"
    if key not in data:
        data[key] = {
            "in_sample": ["sample_anpassungsplaene_N1"],
            "in_gold_standard": []
        }
    else: 
        print(f"{key} already in pdf folder")

# Save the updated JSON
with open(json_path_new, 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=4, ensure_ascii=False)

print("JSON updated successfully.")


JSON updated successfully.
